Демонстрационный проект
1. Загрузка необходимых библиотек и дата-сетов для последующего изучения

Начнём работу с загрузки библиотек и дата-сетов

In [132]:
# импортируем необходимые библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import plotly.express as px

In [92]:
# загрузим файл financial_data.csv и сохраним его в переменную fin_data
fin_data = pd.read_csv('financial_data.csv')
# и сразу проверим, успешно ли произошла загрузка
display(fin_data.head(3))

,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


In [93]:
# то же самое делаем с файлом prolongations.csv, сохраняя его в переменную prolong
prolong = pd.read_csv('prolongations.csv')
display(prolong.head(3))

,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич


In [94]:
# исходя из задачи аналитики, приходим к выводу, что столбец "Причина дубля" не играет ключевой роли в следующем анализе,
# поэтому удаляем этот столбец
fin_data = fin_data.drop(['Причина дубля'], axis=1)

2. Создание таблиц необходимых для дальнейшей аналитики

In [95]:
# в дальнейшем для удобства анализа с сопоставлением информации по разными месяцам
# пойдём путём многомерного массива, и первой сделаем таблицу end_data с булевыми значениями 
# по рабочим отметкам "end" и "стоп", изначально представляющую собой копию fin_data
end_data = fin_data.copy()

# затем пройдёмся циклом по столбцам end_data, начиная с Ноября 2022 (столбец 1) и заканчивая Февралём 2024 (столбец 16)
for col in end_data.columns[1:17]:
    # используя метод apply() и ламбда-функцию в каждом столбце преобразуем ячейки в True, 
    # если в них есть рабочая метка "end" или "стоп", иначе ставиться False
    end_data[col] = end_data[col].apply(lambda x: True if x in ['end', 'стоп'] else False)

# так как строки с id некоторых проектов встречаются несколько раз, то сгруппируем данные по id проекта, 
# применяя агрегирующий метод суммирования, который True посчитает за единицу, а False за ноль
end_data = end_data.groupby(by='id', as_index=False).sum()

# теперь вернём таблицу к булевому виду с помощью всё тех же apply() и лямбда-функции
for col in end_data.columns[1:17]:
    end_data[col] = end_data[col].apply(lambda x: True if x == 1 else False)

# столбец с именами менеджеров в этой таблице не нужен, поэтому удалим его
end_data = end_data.drop(['Account'], axis=1)

# таблица end_data готова

In [96]:
# теперь для многомерного массива нужна булева таблица с рабочей меткой "в ноль", 
# поэтому так же начнёс с создания новой таблицы zero_data как копии fin_data
zero_data = fin_data.copy()

# затем пройдёмся циклом по столбцам zero_data, начиная с Ноября 2022 (столбец 1) и заканчивая Февралём 2024 (столбец 16)
for col in zero_data.columns[1:17]:
    # используя метод apply() и ламбда-функцию в каждом столбце преобразуем ячейки в True, 
    # если в них есть рабочая метка "в ноль", иначе ставиться False
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 'в ноль' else False)

# так как строки с id некоторых проектов встречаются несколько раз, то сгруппируем данные по id проекта, 
# применяя агрегирующий метод суммирования, который True посчитает за единицу, а False за ноль
zero_data = zero_data.groupby(by='id', as_index=False).sum()

# теперь вернём таблицу к булевому виду с помощью всё тех же apply() и лямбда-функции
for col in zero_data.columns[1:17]:
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 1 else False)

# столбец с именами менеджеров в этой таблице не нужен, поэтому удалим его
zero_data = zero_data.drop(['Account'], axis=1)

# таблица zero_data готова

In [97]:
# также необходимо сделать ещё одну булеву таблицу с информацией о месяце закрытия, 
# информация об этом хранится в таблице prolong, поэтому будет необходимо её объединить с таблицей fin_data, 
# но для начала надо сделать одинаковыми названия месяцев в обеих таблицах

# в таблице prolong названия месяцев написаны с маленькой буквы, 
# тогда как в остальных таблицах месяцы в названия столбцов написаны с большоей буквы, 
# поэтому для "стандартизации" названий месяцев напишем функцию first_symbol
def first_symbol(txt):
    # в переменную frst запишем 1-ый символ <строки> txt
    frst = txt[0]
    # этот символ frst сделаем с заглавной буквы с помощьб метода upper()
    frst = frst.upper()
    # и затем соединяем в одну <строку> заглявную букву в переменной frst и срез из txt без 1-го символа
    txt = frst + txt[1:]
    return txt

# теперь с помощью метода apply() и функции first_symbol преобразуем столбец "month"
prolong['month'] = prolong['month'].apply(first_symbol)

###################################################################################################
# создадим ту самую булеву таблицу с информацией о месяце закрытия close_month_data путём 
# объединения таблиц fin_data и prolong
close_month_data = pd.merge(
    left = fin_data,
    right = prolong,
    how = 'left',
    left_on = 'id',
    right_on = 'id'
)

# в таблице close_month_data удалим столбец "AM", который, в принципе, дублирует столбец "Account"
close_month_data = close_month_data.drop(['AM'], axis=1)

###################################################################################################

# пройдёмся циклом по столбцам close_month_data, начиная с Ноября 2022 (столбец 1) и заканчивая Февралём 2024 (столбец 16)
for col in close_month_data.columns[1:17] :
    # а также пройдёмся циклом по всем индексах (строчкам) close_month_data, 
    # т.е. иными словами пройдёмся циклом по всем ячейкам таблицы в столбцах с месяцами
    for string in close_month_data.index :
        # если в соответствующей строчке в столбце "month" (он же месяц закрытия проекта) месяц является названием столбца
        if close_month_data.loc[string, 'month'] == col:
            # то в этой ячейке ставится True
            close_month_data.at[string, col] = True
        else: 
            # иначе в ячейке ставится False
            close_month_data.at[string, col] = False

# так как строки с id некоторых проектов встречаются несколько раз, то сгруппируем данные по id проекта, 
# применяя агрегирующий метод суммирования, который True посчитает за единицу, а False за ноль
close_month_data = close_month_data.groupby(by='id', as_index=False).sum()

# теперь вернём таблицу к булевому виду с помощью всё тех же apply() и лямбда-функции
for col in close_month_data.columns[1:17]:
    close_month_data[col] = close_month_data[col].apply(lambda x: True if x == 1 else False)

# т.к. теперь столбец с информацией о закрытии проектов не нужен, то удаляем столбец "month"
close_month_data = close_month_data.drop(['month'], axis=1)

###################################################################################################

# после группирования и суммирования нужно устранить проблемы со столбцом менеджеров "Account", 
# где <строки> были так же просто "суммированы",
# поэтому пропишем функцию по возвращению нормального ФИО менеджера
def manager_back(txt):
    # разобьём <строку> на <список> <строк> name_list через пробел
    name_list = txt.split(' ')
    # если name_list состоит из 3-х элементов и менее
    if len(name_list) <= 3 :
        # то просто сразу возвращаем нашу исходную <строку>
        return txt
    else:
        # иначе же в переменную t запишем количество символов в 1-ом элементе <списка> name_list (он же должен быть фамилией)
        t = len(name_list[0])
        # затем с помощью f-строк соединяем 3 первых элемента <списка> name_list в одну <строку> name
        name = f'{name_list[0]} {name_list[1]} {name_list[2]}'
        # и, т.к. в 3-ем элементе (индекс 2) из-за суммирования <строк> при группировке к отчеству "приклеилась фамилия", 
        # зная ранее найденное t, можно просто отсечь срез в конце <строки> от отчества фамилию, 
        # и просто получить оставшуюся <строку> с обычным ФИО
        name = name[:-t]
        # которое и вернём в качестве переменной name
        return name

close_month_data['Account'] = close_month_data['Account'].apply(manager_back)

# таблица close_month_data готова

In [98]:
# также для многомерного массива сделаем таблицу pay_data со всеми отгрузками по каждому проекту, 
# она будет как копия fin_data
pay_data = fin_data.copy()

# для преобразования возьмём только столбцы с месяцами, 
# т.е. с Ноября 2022 (столбец номер 1) до Февраля 2024 (столбец номер 16)
for col in pay_data.columns[1:17]:
    # c помощью метода apply() и лямбда-функций сначала преобразуем рабочие метки в NaN
    pay_data[col] = pay_data[col].apply(lambda x: np.nan if x in ['end', 'стоп', 'в ноль'] else x)
    # т.к. в таблице остались пропуски и числа, имеющие тип данных <строка> с двумя нулями после запятой, 
    # заполним пропуски <строками> "0,00" 
    pay_data[col] = pay_data[col].fillna('0,00')
    # и затем удалим неразрывные пробелы '\xa0' с помощью метода replace()
    pay_data[col] = pay_data[col].apply(lambda x: x.replace('\xa0', ''))
    
# при анализе оставшихся <строковых> чисел, обнаружено, что были опечатки, 
# и сотрудники после запятой ставили нули количественно отлично от двух, 
# поэтому пропишем функцию преобразования таких <строк>, 
# чтобы оставалась только целая часть и отсекалось вся часть после запятой
def find_comma(txt):
    # будем находить индекс запятой в <строке> с конца как разница общего количества символов в <строке> и индекс самой запятой
    num = len(txt) - txt.find(',')
    # и отсекать с конца срез после запятой включительно
    txt = txt[:-num]
    return txt

# заново берём тот же цикл с Ноября 2022 (столбец номер 1) до Февраля 2024 (столбец номер 16)
for col in pay_data.columns[1:17]:
    # с помощью функции find_comma отсекаем в <строке> срез после запятой включительно
    pay_data[col] = pay_data[col].apply(find_comma)
    # и преобразуем тип данных из <строки> в <число>
    pay_data[col] = pay_data[col].astype(int)

# так как строки с id некоторых проектов встречаются несколько раз, 
# и причиной дублей были оплаты в несколько траншей, то сгруппируем данные по id проекта, 
# применяя агрегирующий метод суммирования
pay_data = pay_data.groupby(by='id', as_index=False).sum()

# теперь с помощью метода apply() и функции manager_back исправим ошибки в столбце "Account"
pay_data['Account'] = pay_data['Account'].apply(manager_back)

# таблица pay_data готова

In [99]:
# имея таблицу pay_data со всеми отгрузками, теперь можно создать таблицу last_months_data, 
# где будут только суммы отгрузки в последний месяц реализации проекта, учитывая все условия 
# рабочих пометок "стоп", "end" и "в ноль", поэтому изначально сделаем эту таблицу как копию pay_data
last_months_data = pay_data.copy()

# создадим <множество> end_project_set, куда будем заносить id проектов, закончившихся до конца сроков по договору,
# т.е. имеющих рабочие метки "стоп/end" из таблицы end_data, но при окончании проекта по данным из close_month_data 
# будем id проекта исключать их <множества> end_project_set
end_project_set = set()

# пройдёмся циклом по порядку номера месяца, начиная с Ноября 2022 (месяц 1) до Февраля 2024 (месяц 16)
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому индексу,
    for string in last_months_data.index :
        # и заодно сразу находим id проекта
        id_project = last_months_data.loc[string, 'id']
        # если id проекта уже находится в <множестве> end_project_set
        if id_project in end_project_set :
            # то в текущей ячейке (она же месяц "col" и строчка с проектом "string")
            # ставим ноль
            last_months_data.iat[string, col] = 0
            # далее нужно проверить, не закончился ли срок действия договора, 
            # чтобы тогда убрать id проекта из <множества> end_project_set
            if close_month_data.iloc[string, col] == True:
                end_project_set.discard(id_project)
        else:
            # в противном случае проверяем, не имеет ли проект в этом месяце рабочей метки "стоп/end" 
            # об окончании реализации до конца срока по договору
            if end_data.iloc[string, col] == True :
                # если в этой ячейке таблицы в этом месяце будет True, то вносим id проекта в <множество> end_project_set
                end_project_set.add(id_project)
                # и в этой же ячейке таблица last_months_data ставим 0 и прерываем цикл
                last_months_data.iat[string, col] = 0
                continue
            # если рабочей метки об "стоп/end" не было, то проверяем 
            # не заканчивается ли проект в этом месяце
            if close_month_data.iloc[string, col] == False :
                # если не заканчивается (в ячейке False), то ставим 0
                last_months_data.iat[string, col] = 0
            else:
                # если же в ячейке таблицы close_month_data будет True, 
                # то проверим нет ли в этом месяце рабочей метки "в ноль" с помощью таблицы zero_data
                if zero_data.iloc[string, col] == True :
                    # если в таблице zero_data оказывается True, то в ячейку таблицы last_months_data 
                    # вносим отгрузку предыдущего месяца таблицы pay_data
                    last_months_data.iat[string, col] = pay_data.iloc[string, (col-1)]
                # а если в таблице zero_data будет False, то в ячейке last_months_data просто останется значение отгрузки

# таблица last_months_data готова

3. Создание и экспорт файлов аналитического отчёта

Теперь переходим к созданию файлов Excel для отчёта, будем создавать DataFrame и экспортировать его в файл .xslx

In [100]:
# DataFrame с 1-ым коэффициентом по менеджерам ежемесячно (first_coef)
 
# создадим словарь, где <key-порядковый по range> и <value-название месяца>, т.е. {1: 'Ноябрь 2022', 2: 'Декабрь 2022' ... и т.д.}
dict_months = {}
for n in range(1, 17):
    dict_months[n] = pay_data.columns[n]
    
###################################################################################################

# создадим first_prolongation как копию pay_data, чтобы таблица содержала сумму отгрузок по менеджерам после окончаний проектов
first_prolongation = pay_data.copy()

# в Ноябре 2022 (месяц 1) сразу проставим 1-ую пролонгацию как ноль
first_prolongation['Ноябрь 2022'] = first_prolongation['Ноябрь 2022'].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Февраля 2024 (месяц 16)
for col in range(2, 17) :
    # и пройдёмся циклом по всем индексам first_prolongation
    for string in first_prolongation.index :
        # если в соответствующих "координатах" проекта и предыдущего месяца (таблица close_month_data) будет False
        if close_month_data.iloc[string, (col - 1)] == False:
            # то в текущих строке проекта и месяце (найденном по заранее созданному словарю dict_months) ставим ноль
            first_prolongation.at[string, dict_months[col]] = 0
            # если же в этом месяце проект закрывается (True), то значение останется на месте

###################################################################################################

# сгруппируем данные в first_prolongation по менеджерам, считая в месяцах суммы
first_prolongation = first_prolongation.groupby(by='Account', as_index=True).sum()

# удалим лишние столбцы
first_prolongation = first_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

###################################################################################################

# для подсчёта 1-го коэффициента сделаем таблицу first_coef, как копию таблицы first_prolongation
first_coef = first_prolongation.copy()

# и создадим словарь coef_sum_dict, в котором по ключу месяца значением будет сумма отгрузки за прошлый месяц, 
# т.е. {'Январь 2023': <сумма отгрузки за Декабрь 2022>, 'Февраль 2023': <сумма отгрузки за Январь 2023> ... и т.д.}
coef_sum_dict = {}

# для заполнения словаря coef_sum_dict пройдёмся циклом по номерам месяцев, 
# от Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13) - они будут предыдущими месяцами
for last_month in range(2, 14) :
    # зная номер предыдущего месяца, будем находить столбец следующего (целевого) месяца
    goal_month = last_months_data.columns[last_month + 1]
    # затем находить сумму отгрузки за предыдущий месяц coef_sum
    coef_sum = last_months_data[last_months_data.columns[last_month]].sum()
    # и по ключу целевого месяца goal_month в словарь coef_sum_dict 
    # будем заносить значение суммы отгрузки за предыдущий месяц coef_sum
    coef_sum_dict[goal_month] = coef_sum

# пройдёмся циклом по каждому месяцу таблицы first_coef
for col in first_coef.columns[0:13] :
    # и пройдёмся циклом по каждому индексу first_coef (в роли индексов у нас уже ФИО менеджеров)
    for string in first_coef.index:
        # в итоге для каждого менеджера в каждом месяце рассчитываем отношение суммы пролонгации у конкретного менеджера
        # к сумме отгрузки всех закрывающихся в предудыщем месяце проектов, округляя до 3-х знаков после запятой
        first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
        
# DataFrame с 1-ым коэффициентом по менеджерам ежемесячно (first_coef) готов

C:\Users\azott\AppData\Local\Temp\ipykernel_7332\560826496.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.06' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_7332\560826496.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.359' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_7332\560826496.py:60: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.208' has dtype incompatible with int64, pl

In [101]:
# экспортируем DataFrame с 1-ым коэффициентом по менеджерам ежемесячно (first_coef) в Excel
first_coef.to_excel(excel_writer = 'Коэффициенты пролонгации по менеджерам.xlsx', sheet_name = '1-ый коэффициент', index=True)

In [102]:
# DataFrame со 2-ым коэффициентом по менеджерам ежемесячно (second_coef)

# создадим словарь, где <key-порядковый по range> и <value-название месяца>, т.е. {1: 'Ноябрь 2022', 2: 'Декабрь 2022' ... и т.д.}
dict_months = {}
for n in range(1, 17):
    dict_months[n] = pay_data.columns[n]
    
###################################################################################################

# создадим second_prolongation как копию pay_data, чтобы таблица содержала сумму отгрузок по менеджерам после окончаний проектов
second_prolongation = pay_data.copy()

# в Ноябре 2022 (месяц 1) и Декабре 2022 (месяц 2) сразу проставим 2-ую пролонгацию как ноль
second_prolongation['Ноябрь 2022'] = second_prolongation['Ноябрь 2022'].apply(lambda x: 0)
second_prolongation['Декабрь 2022'] = second_prolongation['Декабрь 2022'].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Января 2023 (месяц 3) до Февраля 2024 (месяц 16)
for col in range(3, 17) :
    # и пройдёмся циклом по всем индексам second_prolongation
    for string in second_prolongation.index :
        # если в соответствующих "координатах" проекта и два месяца назад (таблица close_month_data) будет False,
        # а также в таблице pay_data месяц назад будет ноль
        if close_month_data.iloc[string, (col - 2)] == True and pay_data.iloc[string, (col - 1)] == 0:
            # то в текущих строке проекта и месяце оставляем всё, как было
            continue
        else:
            # иначе, если оба условия не выполняются, то в текущих строке проекта и месяце 
            # (найденном по заранее созданному словарю dict_months) ставим ноль
            second_prolongation.at[string, dict_months[col]] = 0
            
###################################################################################################

# сгруппируем данные в second_prolongation по менеджерам, считая в месяцах суммы
second_prolongation = second_prolongation.groupby(by='Account', as_index=True).sum()

# удалим лишние столбцы
second_prolongation = second_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

###################################################################################################

# для подсчёта 2-го коэффициента сделаем таблицу second_coef, как копию таблицы second_prolongation
second_coef = second_prolongation.copy()

# и создадим словарь coef_sum_dict_2, в котором по ключу месяца значением будет сумма отгрузки за два месяца назад 
# при условии отсутствия отгрузок месяц назад, т.е. 
# {'Январь 2023': <сумма отгрузки заканчивающихся проектов за Ноябрь 2022, если нет отгрузок в Декабре 2022>,
# 'Февраль 2023': <сумма отгрузки заканчивающихся проектов за Декабрь 2022, если нет отгрузок в Январее 2023> ... и т.д.}
coef_sum_dict_2 = {}

# для заполнения словаря coef_sum_dict_2 пройдёмся циклом по номерам месяцев, 
# от Ноября 2022 (месяц 1) до Ноября 2023 (месяц 13) - они будут позапрошлыми месяцами
for before_last_month in range(1, 14) :
    # зная номер позапрошлого месяца, будем находить столбец текущего (целевого) месяца
    goal_month = last_months_data.columns[before_last_month + 2]
    # затем проверять является ли позапрошлый месяц завершающим для проекта (по таблице close_month_data)
    # и были ли отгрузки в предыдущем месяце по нулям (по таблице pay_data),
    # предварительно для подсчёта создадим переменную общей суммы искомой отгрузки sum_for_goal
    sum_for_goal = 0
    for string in close_month_data.index :
        if close_month_data.iloc[string, before_last_month] == True and pay_data.iloc[string, before_last_month + 1] == 0:
            sum_for_goal += pay_data.iloc[string, before_last_month]
    # затем эту сумму sum_for_goal вносим как значение в словарь coef_sum_dict_2, 
    # где ключом будет целевой месяц goal_month
    coef_sum_dict_2[goal_month] = sum_for_goal

# пройдёмся циклом по каждому месяцу таблицы second_coef
for col in second_coef.columns[0:13] :
    # и пройдёмся циклом по каждому индексу second_coef (в роли индексов у нас уже ФИО менеджеров)
    for string in second_coef.index:
        # в итоге для каждого менеджера в каждом месяце рассчитываем отношение суммы пролонгации у конкретного менеджера
        # к сумме отгрузки всех закрывающихся в предудыщем месяце проектов, округляя до 3-х знаков после запятой
        second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict_2[col] , 3)
        
# DataFrame со 2-ым коэффициентом по менеджерам ежемесячно (second_coef) готов

C:\Users\azott\AppData\Local\Temp\ipykernel_7332\3886687365.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.184' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict_2[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_7332\3886687365.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.026' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict_2[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_7332\3886687365.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.346' has dtype incompatible wi

In [103]:
# экспортируем DataFrame со 2-ым коэффициентом по менеджерам ежемесячно (second_coef) в Excel
second_coef.to_excel(excel_writer = '2-ой коэффициент пролонгации по менеджерам.xlsx', sheet_name = '2-ой коэффициент', index=True)

Лист в Excel "Менеджеры по месяцам" будет делать путём добавления в файл "Коэффициенты пролонгации по менеджерам.xlsx" таблицы из файла "2-ой коэффициент пролонгации по менеджерам.xlsx", и далее переименования листа в "Менеджеры по месяцам"

Для создания в отчёте листа "Менеджеры за год" сделаем DataFrame "managers_for_year", который потом экспортируем в Excel "Менеджеры за год.xlsx".
Сам DataFrame "managers_for_year" сделаем из нескольких объектов Series и DataFrame

In [104]:
# для DataFrame "managers_for_year" сделаем Series "manag_projects_for_first"
# с информацией листа "Менеджеры за год" таблицы Excel 
# во 2-ом столбце "к пролонгации" первого месяца по менеджерам

# для расчёта количества проектов к пролонгации в 1-ый месяц сделаем копию close_month_data
manag_projects_for_first = close_month_data.copy()

# и в новой таблице manag_projects_for_first все булевы значения заменим на нули
for col in manag_projects_for_first.columns[1:17]:
    manag_projects_for_first[col] = manag_projects_for_first[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13)
for col in range(2, 14) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True
        if close_month_data.iloc[string, col] == True:
            # то в таблице manag_projects_for_first на следующий месяц ставим 1
            manag_projects_for_first.iat[string, (col + 1)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_for_first = manag_projects_for_first.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_for_first['сумма к 1-ой пролонгации'] = manag_projects_for_first.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов к 1-ой пролонгации
manag_projects_for_first = manag_projects_for_first.groupby('Account')['сумма к 1-ой пролонгации'].sum()

# Series "manag_projects_for_first" для 2-го столбца "к пролонгации" первого месяца по менеджерам готов

In [105]:
# для DataFrame "managers_for_year" сделаем Series "manag_projects_for_second"
# с информацией листа "Менеджеры за год" таблицы Excel 
# в 5-ом столбце "к пролонгации" второго месяца по менеджерам

# для расчёта количества проектов к пролонгации во 2-ой месяц сделаем копию close_month_data
manag_projects_for_second = close_month_data.copy()

# и в новой таблице manag_projects_for_second все булевы значения заменим на нули
for col in manag_projects_for_second.columns[1:17]:
    manag_projects_for_second[col] = manag_projects_for_second[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Ноября 2022 (месяц 1) до Октября 2023 (месяц 12)
for col in range(1, 13) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и в текущих строчке и в следующем месяце в таблице pay_data будет 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] == 0 :
            # то в таблице all_projects_for_first на два месяца вперёд ставим 1
            manag_projects_for_second.iat[string, (col + 2)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_for_second = manag_projects_for_second.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_for_second['сумма ко 2-ой пролонгации'] = manag_projects_for_second.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов ко 2-ой пролонгации
manag_projects_for_second = manag_projects_for_second.groupby('Account')['сумма ко 2-ой пролонгации'].sum()

# Series "manag_projects_for_second" для 5-го столбца "к пролонгации" второго месяца по менеджерам готов

In [106]:
# для DataFrame "managers_for_year" сделаем Series "manag_projects_first_prolong"
# с информацией листа "Менеджеры за год" таблицы Excel 
# в 3-ем столбце "пролонгировано" первого месяца по менеджерам

# для расчёта количества проектов, пролонгированных в 1-ый месяц, сделаем копию close_month_data
manag_projects_first_prolong = close_month_data.copy()

# и в новой таблице manag_projects_first_prolong все булевы значения заменим на нули
for col in manag_projects_first_prolong.columns[1:17]:
    manag_projects_first_prolong[col] = manag_projects_first_prolong[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13)
for col in range(2, 14) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и если в текущей строчке и следующем месяце в таблице pay_data будет не 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] != 0 :
            # то в таблице manag_projects_first_prolong на следующий месяц ставим 1
            manag_projects_first_prolong.iat[string, (col + 1)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_first_prolong = manag_projects_first_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_first_prolong['сумма пролонгировано в 1-ый месяц'] = manag_projects_first_prolong.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов к 1-ой пролонгации
manag_projects_first_prolong = manag_projects_first_prolong.groupby('Account')['сумма пролонгировано в 1-ый месяц'].sum()

# Series "manag_projects_first_prolong" для 3-го столбца "пролонгировано" первого месяца по менеджерам готов

In [107]:
# для DataFrame "managers_for_year" сделаем Series "manag_projects_second_prolong"
# с информацией листа "Менеджеры за год" таблицы Excel 
# в 6-ом столбце "пролонгировано" второго месяца по менеджерам

# для расчёта количества проектов, пролонгированных во 2-ой месяц, сделаем копию close_month_data
manag_projects_second_prolong = close_month_data.copy()

# и в новой таблице manag_projects_second_prolong все булевы значения заменим на нули
for col in manag_projects_second_prolong.columns[1:17]:
    manag_projects_second_prolong[col] = manag_projects_second_prolong[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Ноября 2022 (месяц 1) до Октября 2023 (месяц 12)
for col in range(1, 13) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и если в текущей строчке и следующем месяце в таблице pay_data будет 0, 
        # но ещё месяцем позже в таблице pay_data будет не 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] == 0 and pay_data.iloc[string, (col + 2)] != 0:
            # то в таблице manag_projects_second_prolong на два месяца вперёд ставим 1
            manag_projects_second_prolong.iat[string, (col + 2)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_second_prolong = manag_projects_second_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_second_prolong['сумма пролонгировано во 2-ой месяц'] = manag_projects_second_prolong.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов к 1-ой пролонгации
manag_projects_second_prolong = manag_projects_second_prolong.groupby('Account')['сумма пролонгировано во 2-ой месяц'].sum()

# Series "manag_projects_second_prolong" для 6-го столбца "пролонгировано" второго месяца по менеджерам готов

In [108]:
# для DataFrame "managers_for_year" сделаем другой DataFrame "first_coef_manag" 
# (где нас будет интересовать столбец "1-ый коэффициент среднее за год")
# с информацией листа "Менеджеры за год" таблицы Excel 
# в 4-ом столбце "коэффициент" первого месяца по менеджерам

# для подсчёта 1-го коэффициента по каждому менеджеру за год 
# сделаем таблицу first_coef_manag как копию таблицы first_coef (1-ый коэффициент по каждому менеджеру ежемесячно)
first_coef_manag = first_coef.copy()

# в новой таблице сделаем новый признак "1-ый коэффициент среднее за год", 
# который будем рассчитывать как среднее по всем месяцам, округлённое до 3-х знаков после запятой
first_coef_manag['1-ый коэффициент среднее за год'] = round( first_coef_manag.mean(numeric_only=True, axis=1) , 3)

# и далее пройдёмся циклом по всем месяцам таблицы first_coef_manag, т.е. с Января 2023 по Декабрь 2023
for col in first_coef_manag.columns[0:12]:
    # и в каждой итерации будем удалять этот столбец, чтобы остался DataFrame с ФИО менеджеров и их годовым 1-ым коэффициентом
    first_coef_manag = first_coef_manag.drop([col], axis=1)
    
# DataFrame "first_coef_manag" для 4-го столбца "коэффициент" первого месяца по менеджерам готов

In [109]:
# для DataFrame "managers_for_year" сделаем другой DataFrame "second_coef_manag"
# (где нас будет интересовать столбец "2-ой коэффициент среднее за год")
# с информацией листа "Менеджеры за год" таблицы Excel 
# в 7-ом столбце "коэффициент" второго месяца по менеджерам

# для подсчёта 2-го коэффициента по каждому менеджеру за год 
# сделаем таблицу second_coef_manag как копию таблицы second_coef (2-ой коэффициент по каждому менеджеру ежемесячно)
second_coef_manag = second_coef.copy()

# в новой таблице сделаем новый признак "2-ой коэффициент среднее за год", 
# который будем рассчитывать как среднее по всем месяцам, округлённое до 3-х знаков после запятой
second_coef_manag['2-ой коэффициент среднее за год'] = round( second_coef_manag.mean(numeric_only=True, axis=1) , 3)

# и далее пройдёмся циклом по всем месяцам таблицы second_coef_manag, т.е. с Января 2023 по Декабрь 2023
for col in second_coef_manag.columns[0:12]:
    # и в каждой итерации будем удалять этот столбец, чтобы остался DataFrame с ФИО менеджеров и их годовым 2-ым коэффициентом
    second_coef_manag = second_coef_manag.drop([col], axis=1)
    
# DataFrame "second_coef_manag" для 7-го столбца "коэффициент" второго месяца по менеджерам готов

In [110]:
# теперь для создания DataFrame "managers_for_year" всё готово
# во всех созданных ранее Series и DataFrame всё отсортировано в алфавитном порядке по фамилии менеджера, 
# поэтому для столбца "Менеджер" возьмём <список> через индексы любой таблицы, например manag_projects_for_first, 
# а остальные столбцы будем заполнять данными из созданных Series и DataFrame, переводя их в тип данных <список>
managers_for_year = pd.DataFrame({
    'Менеджер': manag_projects_for_first.index,
    'к пролонгации в 1-ый месяц': list(manag_projects_for_first),
    'пролонгировано в 1-ый месяц': list(manag_projects_first_prolong),
    'коэффициент 1-ой пролонгации': list(first_coef_manag['1-ый коэффициент среднее за год']),
    'к пролонгации во 2-ой месяц': list(manag_projects_for_second),
    'пролонгировано во 2-ой месяц': list(manag_projects_second_prolong),
    'коэффициент 2-ой пролонгации': list(second_coef_manag['2-ой коэффициент среднее за год'])
    })

# DataFrame "managers_for_year" для создания в отчёте листа "Менеджеры за год" готов

In [111]:
# экспортируем DataFrame с информацией о работе менеджеров за год в Excel
managers_for_year.to_excel(excel_writer = 'Менеджеры за год.xlsx', sheet_name = 'Менеджеры за год', index=True)

Для создания в отчёте листа "Весь отдел" сделаем DataFrame "all_dept", который потом экспортируем в Excel "Весь отдел.xlsx".
Сам DataFrame "all_dept" сделаем из нескольких словарей

In [112]:
# для DataFrame "all_dept" сделаем <словарь> "all_projects_for_first"
# с информацией листа "Весь отдел" таблицы Excel 
# во 2-ом столбце "к пролонгации" первого месяца по всему отделу

# для подсчёта количества проектов к пролонгации в 1-ый месяц на каждый месяц по всему отделу
# создадим пустой словарь all_projects_for_first, у которого в дальнейшем будет вид 
# {'Январь 2023': <кол-во проектов к пролонгации в 1-ый месяц>, 'Февраль 2023': <кол-во проектов к пролонгации в 1-ый месяц>, ... и т.д.}
all_projects_for_first = {}

# в начале сделаем то же самое, что и с подсчётом кол-ва проектов к пролонгации по менеджерам, 
# а именно сделаем копию close_month_data (как делали manag_projects_for_first)
months_projects_for_first = close_month_data.copy()

# и в новой таблице months_projects_for_first все булевы значения заменим на нули
for col in months_projects_for_first.columns[1:17]:
    months_projects_for_first[col] = months_projects_for_first[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13)
for col in range(2, 14) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True
        if close_month_data.iloc[string, col] == True:
            # то в таблице months_projects_for_first на следующий месяц ставим 1
            months_projects_for_first.iat[string, (col + 1)] = 1

# удалим лишние столбцы с id проекта и месяцами         
months_projects_for_first = months_projects_for_first.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# а далее (в отличие от преобразований с manag_projects_for_first) пройдёмся циклом по каждому месяцу, 
# находя сумму в каждом столбце, и эту сумму, как значение по ключу месяца, будем записывать в словарь all_projects_for_first
for month in months_projects_for_first.columns[0:12] :
    all_projects_for_first[month] = months_projects_for_first[month].sum()
    
# <словарь> "all_projects_for_first" для 2-го столбца "к пролонгации" первого месяца по всему отделу готов


In [113]:
# для DataFrame "all_dept" сделаем <словарь> "all_projects_for_second"
# с информацией листа "Весь отдел" таблицы Excel 
# в 5-ом столбце "к пролонгации" второго месяца по всему отделу

# для подсчёта количества проектов к пролонгации во 2-ой месяц на каждый месяц по всему отделу
# создадим пустой словарь all_projects_for_second, у которого в дальнейшем будет вид 
# {'Январь 2023': <кол-во проектов к пролонгации во 2-ой месяц>, 'Февраль 2023': <кол-во проектов к пролонгации во 2-ой месяц>, ... и т.д.}
all_projects_for_second = {}

# в начале сделаем то же самое, что и с подсчётом кол-ва проектов к пролонгации по менеджерам, 
# а именно сделаем копию close_month_data (как делали manag_projects_for_second)
months_projects_for_second = close_month_data.copy()

# и в новой таблице months_projects_for_second все булевы значения заменим на нули
for col in months_projects_for_second.columns[1:17]:
    months_projects_for_second[col] = months_projects_for_second[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Ноября 2022 (месяц 1) до Октября 2023 (месяц 12)
for col in range(1, 13) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и в текущих строчке и в следующем месяце в таблице pay_data будет 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] == 0 :
            # то в таблице all_projects_for_first на два месяца вперёд ставим 1
            months_projects_for_second.iat[string, (col + 2)] = 1

# удалим лишние столбцы с id проекта и месяцами         
months_projects_for_second = months_projects_for_second.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# а далее (в отличие от преобразований с manag_projects_for_second) пройдёмся циклом по каждому месяцу, 
# находя сумму в каждом столбце, и эту сумму, как значение по ключу месяца, будем записывать в словарь months_projects_for_second
for month in months_projects_for_second.columns[0:12] :
    all_projects_for_second[month] = months_projects_for_second[month].sum()
    
# <словарь> "all_projects_for_second" для 5-го столбца "к пролонгации" второго месяца по всему отделу готов

In [114]:
# для DataFrame "all_dept" сделаем <словарь> "all_projects_first_prolong"
# с информацией листа "Весь отдел" таблицы Excel 
# в 3-ем столбце "пролонгировано" первого месяца по всему отделу

# для подсчёта количества пролонгированных проектов в 1-ый месяц на каждый месяц по всему отделу
# создадим пустой словарь all_projects_first_prolong, у которого в дальнейшем будет вид 
# {'Январь 2023': <кол-во пролонгированных проектов в 1-ый месяц>, 'Февраль 2023': <кол-во пролонгированных проектов в 1-ый месяц>, ... и т.д.}
all_projects_first_prolong = {}

# в начале сделаем то же самое, что и с подсчётом кол-ва пролонгированных проектов по менеджерам, 
# а именно сделаем копию close_month_data (как делали manag_projects_first_prolong)
months_projects_first_prolong = close_month_data.copy()

# и в новой таблице months_projects_first_prolong все булевы значения заменим на нули
for col in months_projects_first_prolong.columns[1:17]:
    months_projects_first_prolong[col] = months_projects_first_prolong[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13)
for col in range(2, 14) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и если в текущей строчке и следующем месяце в таблице pay_data будет не 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] != 0 :
            # то в таблице months_projects_first_prolong на следующий месяц ставим 1
            months_projects_first_prolong.iat[string, (col + 1)] = 1

# удалим лишние столбцы с id проекта и месяцами         
months_projects_first_prolong = months_projects_first_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# а далее (в отличие от преобразований с manag_projects_first_prolong) пройдёмся циклом по каждому месяцу, 
# находя сумму в каждом столбце, и эту сумму, как значение по ключу месяца, будем записывать в словарь all_projects_first_prolong
for month in months_projects_first_prolong.columns[0:12] :
    all_projects_first_prolong[month] = months_projects_first_prolong[month].sum()
    
# <словарь> "all_projects_first_prolong" для 3-го столбца "пролонгировано" первого месяца по всему отделу готов

In [115]:
# для DataFrame "all_dept" сделаем <словарь> "all_projects_second_prolong"
# с информацией листа "Весь отдел" таблицы Excel 
# в 6-ом столбце "пролонгировано" второго месяца по всему отделу

# для подсчёта количества пролонгированных проектов во 2-ой месяц на каждый месяц по всему отделу
# создадим пустой словарь all_projects_second_prolong, у которого в дальнейшем будет вид 
# {'Январь 2023': <кол-во пролонгированных проектов во 2-ой месяц>, 'Февраль 2023': <кол-во пролонгированных проектов во 2-ой месяц>, ... и т.д.}
all_projects_second_prolong = {}

# в начале сделаем то же самое, что и с подсчётом кол-ва пролонгированных проектов по менеджерам, 
# а именно сделаем копию close_month_data (как делали manag_projects_second_prolong)
months_projects_second_prolong = close_month_data.copy()

# и в новой таблице months_projects_second_prolong все булевы значения заменим на нули
for col in months_projects_second_prolong.columns[1:17]:
    months_projects_second_prolong[col] = months_projects_second_prolong[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Ноября 2022 (месяц 1) до Октября 2023 (месяц 12)
for col in range(1, 13) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и если в текущей строчке и следующем месяце в таблице pay_data будет не 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] == 0 and pay_data.iloc[string, (col + 2)] != 0:
            # то в таблице manag_projects_first_prolong на следующий месяц ставим 1
            months_projects_second_prolong.iat[string, (col + 1)] = 1

# удалим лишние столбцы с id проекта и месяцами         
months_projects_second_prolong = months_projects_second_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# а далее (в отличие от преобразований с manag_projects_second_prolong) пройдёмся циклом по каждому месяцу, 
# находя сумму в каждом столбце, и эту сумму, как значение по ключу месяца, будем записывать в словарь all_projects_second_prolong
for month in months_projects_second_prolong.columns[0:12] :
    all_projects_second_prolong[month] = months_projects_second_prolong[month].sum()
    
# <словарь> "all_projects_second_prolong" для 6-го столбца "пролонгировано" второго месяца по всему отделу готов

In [116]:
# для DataFrame "all_dept" сделаем <словарь> "first_coef_dept"
# с информацией листа "Весь отдел" таблицы Excel 
# в 4-ом столбце "коэффициент" первого месяца по всему отделу

# создадим пустой словарь first_coef_dept, где ключом будет месяц, 
# а значением среднее по 1-ым коэффициентам всех менеджеров, т.е. будет иметь вид 
# {'Январь 2023': <среднее значение 1-го коэффициента всех менеджеров>, 
# 'Февраль 2023': <среднее значение 1-го коэффициента всех менеджеров>, ... и т.д.}
first_coef_dept = {}

# для этого у нас уже есть ранее созданная таблица first_coef с ежемесячными 1-ыми коэффициентами по каждому менеджеру,
# поэтому пройдёмся циклом по каждому месяцу этой таблицы
for month in first_coef.columns:
    # и будем заносить в словарь first_coef_dept по ключу месяца среднее значение 1-го коэффициета, округлённое до 3-х знаков после запятой
    first_coef_dept[month] = round( first_coef[month].mean() , 3)
    
# <словарь> "first_coef_dept" для 4-го столбца "коэффициент" первого месяца по всему отделу готов

In [117]:
# для DataFrame "all_dept" сделаем <словарь> "second_coef_dept"
# с информацией листа "Весь отдел" таблицы Excel 
# в 7-ом столбце "коэффициент" второго месяца по всему отделу

# создадим пустой словарь second_coef_dept, где ключом будет месяц, 
# а значением среднее по 2-ым коэффициентам всех менеджеров, т.е. будет иметь вид 
# {'Январь 2023': <среднее значение 2-го коэффициента всех менеджеров>, 
# 'Февраль 2023': <среднее значение 2-го коэффициента всех менеджеров>, ... и т.д.}
second_coef_dept = {}

# для этого у нас уже есть ранее созданная таблица second_coef с ежемесячными 2-ыми коэффициентами по каждому менеджеру,
# поэтому пройдёмся циклом по каждому месяцу этой таблицы
for month in second_coef.columns:
    # и будем заносить в словарь second_coef_dept по ключу месяца среднее значение 2-го коэффициета, округлённое до 3-х знаков после запятой
    second_coef_dept[month] = round( second_coef[month].mean() , 3)
    
# <словарь> "second_coef_dept" для 7-го столбца "коэффициент" второго месяца по всему отделу готов

In [118]:
# теперь для создания DataFrame "all_dept" всё готово
# во всех созданных ранее <словарях> всё отсортировано хронологически по ключам месяцев от Января 2023 до Декабря 2023, 
# поэтому для столбца "Месяц" возьмём <список> через индексы любой таблицы, например months_projects_for_first, 
# а остальные столбцы будем заполнять данными из значений созданных <словарей> с помощью метода values(), переводя их в тип данных <список>
all_dept = pd.DataFrame({
    'Месяц': months_projects_for_first.columns[0:12],
    'к пролонгации 1-ый месяц': list(all_projects_for_first.values()),
    'пролонгировано 1-ый месяц': list(all_projects_first_prolong.values()),
    'коэффициент 1-ой пролонгации': list(first_coef_dept.values()),
    'к пролонгации 2-ой месяц': list(all_projects_for_second.values()),
    'пролонгировано 2-ой месяц': list(all_projects_second_prolong.values()),
    'коэффициент 2-ой пролонгации': list(second_coef_dept.values())
})

# DataFrame "all_dept" для создания в отчёте листа "Весь отдел" готов

In [119]:
# экспортируем DataFrame с информацией о работе всего отдела за год в Excel
all_dept.to_excel(excel_writer = 'Весь отдел.xlsx', sheet_name = 'Весь отдел', index=True)

4. Визуализация данных для аналитического отчёта

In [177]:
managers_for_year['Пролонгировано 1 месяц'] = managers_for_year['пролонгировано в 1-ый месяц']
managers_for_year['Не пролонгировано 1 месяц'] = managers_for_year['к пролонгации в 1-ый месяц'] - managers_for_year['пролонгировано в 1-ый месяц']

fig = px.bar(
    data_frame = managers_for_year,
    x = ['Пролонгировано 1 месяц', 'Не пролонгировано 1 месяц'],
    y = 'Менеджер',
    height = 400,
    width = 800,
    title = 'Эффективность работы менеджеров в 1-ый месяц'
)
fig.show()

managers_for_year = managers_for_year.drop(['Пролонгировано 1 месяц', 'Не пролонгировано 1 месяц'], axis=1)

In [178]:
managers_for_year['Пролонгировано 2 месяц'] = managers_for_year['пролонгировано во 2-ой месяц']
managers_for_year['Не пролонгировано 2 месяц'] = managers_for_year['к пролонгации во 2-ой месяц'] - managers_for_year['пролонгировано во 2-ой месяц']

fig = px.bar(
    data_frame = managers_for_year,
    x = ['Пролонгировано 2 месяц', 'Не пролонгировано 2 месяц'],
    y = 'Менеджер',
    height = 400,
    width = 800,
    title = 'Эффективность работы менеджеров во 2-ой месяц'
)
fig.show()

managers_for_year = managers_for_year.drop(['Пролонгировано 2 месяц', 'Не пролонгировано 2 месяц'], axis=1)

In [173]:
fig = px.pie(
    data_frame = managers_for_year,
    names = 'Менеджер',
    values = 'к пролонгации в 1-ый месяц',
    height = 450,
    width = 700,
    title = 'Процент проектов для каждого менеджера, которые нужно пролонгировать'
)
fig.show()

In [ ]:
# построим тепловую карту по каждому менеджеру